# Mini-model języka na nazwach chorób
### Sympozjum „Nowoczesne technologie w pediatrii" · Kamil Jędryczek

Ten notatnik pokazuje **ten sam mechanizm co ChatGPT**, tylko w miniaturze (~80 tys. parametrów): sieć przewiduje **następną literę** w łacińskiej nazwie choroby.

**Uruchomienie:** menu `Środowisko wykonawcze` → `Uruchom wszystko` (Ctrl+F9). Trening na CPU zajmuje ok. **15 sekund** (pierwsze uruchomienie w Colabie — dłużej, bo instaluje pakiety).

*Aplikację dla sali (telefon, bez logowania) uruchamia prelegent skryptem `notebook/uruchom_demo.py` — nie z tego notatnika.*

In [ ]:
# ── USTAWIENIA ────────────────────────────────────────────────
ZBIOR = "choroby"          # można: "leki", "dinozaury"
LICZBA_EPOK = 12           # dobrane eksperymentami (szybko + sensowne nazwy)
TEMPO_UCZENIA = 0.003
TEMPERATURA = 0.4          # niżej = ostrożniej, wyżej = bardziej „kreatywnie"
ZIARNO = 42                # stałe ziarno → ten sam model przy każdym uruchomieniu

REPO = "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/data"
DANE = {
    "choroby": f"{REPO}/choroby.csv",
    "leki": f"{REPO}/leki.csv",
    "dinozaury": "https://raw.githubusercontent.com/junosuarez/dinosaurs/master/dinosaurs.csv",
}

# Wspólny moduł ze skryptem uruchom_demo.py (na Colabie pobiera się sam, jeśli go nie ma).
import sys
from pathlib import Path
_kandydaci = [Path.cwd(), Path.cwd() / "notebook", Path.cwd().parent / "notebook"]
_katalog = next((k for k in _kandydaci if (k / "demo_model.py").exists()), None)
if _katalog is None:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/notebook/demo_model.py",
        "demo_model.py",
    )
    _katalog = Path.cwd()
sys.path.insert(0, str(_katalog))

from demo_model import (
    generuj,
    przygotuj_dane,
    trenuj,
    ustaw_ziarna,
    wczytaj_nazwy,
    zbuduj_model,
)

ustaw_ziarna(ZIARNO)

In [ ]:
# ── WCZYTAJ NAZWY ─────────────────────────────────────────────
nazwy, zrodlo = wczytaj_nazwy(ZBIOR, ziarno=ZIARNO, dane=DANE)
print(f"Wczytano {len(nazwy)} nazw z: {zrodlo}")
print("Przykłady:", ", ".join(nazwy[:6]))

## Przygotowanie danych

Każdą nazwę opakowujemy w znaki sterujące:
- `%` na początku — **START** („zacznij pisać"),
- `!` na końcu — **STOP** („koniec nazwy").

Przykład: `pneumonia` → `%pneumonia!`. Model uczy się: *dany prefiks → jaka litera następna?*

In [ ]:
# ── SŁOWNIK I MACIERZE X, Y ─────────────────────────────────────
X, Y, znak2id, id2znak, znaki = przygotuj_dane(nazwy)
WIELKOSC_SLOWNIKA = len(znaki) + 1
print(f"Tokenów: {WIELKOSC_SLOWNIKA} ({''.join(znaki)})")
print(f"Kształt danych: {X.shape[0]} nazw × {X.shape[1]} pozycji")

## Model

Trzy warstwy Keras: **Embedding(64)** → **GRU(128)** z dropout 0.2 → **softmax** (rozkład następnej litery).

To ta sama idea co w dużych modelach: kontekst → prawdopodobieństwa → wybór następnego tokenu — tu tokenem jest pojedyncza litera.

In [ ]:
# ── TRENING ───────────────────────────────────────────────────
model = zbuduj_model(WIELKOSC_SLOWNIKA, tempo=TEMPO_UCZENIA, ziarno=ZIARNO)
historia = trenuj(model, X, Y, epoki=LICZBA_EPOK)

In [ ]:
# ── JAK SPADA BŁĄD? ───────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.5))
plt.plot(historia.history["loss"], label="błąd treningu")
plt.xlabel("Epoka")
plt.ylabel("Loss")
plt.title("Im niżej, tym lepiej model trafia następną literę")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Generowanie

1. Zaczynamy od `%`.
2. Model podaje rozkład liter — **losujemy** jedną (temperatura = „odwaga" losowania).
3. Dopisujemy literę i powtarzamy, aż wypadnie `!`.

Model **zawsze** coś wybiera — nigdy nie mówi „nie wiem". Na końcu sprawdzamy, czy wylosowana nazwa **już była w zbiorze treningowym**.

In [ ]:
# ── JEDNA NAZWA ───────────────────────────────────────────────
ZNANE = set(nazwy)

def czy_nowa(nazwa: str) -> str:
    if nazwa in ZNANE:
        return f'⚠️  „{nazwa}” — już była w zbiorze treningowym (model ją odtworzył)'
    return f'✨ „{nazwa}” — nie ma w zbiorze (model coś wymyślił)'

nazwa, _ = generuj(model, znak2id, id2znak, temperatura=TEMPERATURA)
pierwsza_nazwa = nazwa
print("Wynik:", nazwa)
print(czy_nowa(nazwa))

In [ ]:
# ── 10 NAZW ───────────────────────────────────────────────────
lista_nazw = []
nowe = 0
for _ in range(10):
    wynik, _ = generuj(model, znak2id, id2znak, temperatura=TEMPERATURA)
    lista_nazw.append(wynik)
    znacznik = "⚠️ w zbiorze" if wynik in ZNANE else "✨ nowa"
    if wynik not in ZNANE:
        nowe += 1
    print(f"  • {wynik:<36} {znacznik}")
print(f"\n{nowe}/10 nazw nie występuje w zbiorze treningowym.")